# Data Correction Notebook

This notebook performs the following corrections on KCET_split.csv:
1. Remove Round 0 (Mock Allotment Round) rows
2. Remove MISSING_COLLEGE rows
3. Create College Code to College Name mapping (JSON) for 2024 data

**Important**: Before deletion, we analyze the data to understand what we're removing.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import json
import os

# Set paths
data_path = r'D:\Major Project\college-predictor\Version2\data\KCET_split.csv'
output_dir = r'D:\Major Project\college-predictor\Version2\data\processed_data'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Load the data
print("Loading data...")
df = pd.read_csv(data_path)
print(f"Original dataset shape: {df.shape}")
print(f"Total rows: {len(df):,}")
print(f"\nColumns: {df.columns.tolist()}")

Loading data...
Original dataset shape: (300689, 12)
Total rows: 300,689

Columns: ['College_Code', 'College_Name', 'Category', 'Branch', 'Cutoff_Rank', 'Year', 'Round', 'Exam_Type', 'Rank_Scaled', 'Branch_Norm', 'College_Clean', 'College_Final']


## Step 1: Analyze Round 0 Data

Before removing Round 0 rows, let's understand:
- How many Round 0 rows exist?
- Distribution across years
- What percentage of data will be removed?

In [2]:
# Analyze Round 0 data
print("=" * 70)
print("ROUND 0 (MOCK ALLOTMENT) ANALYSIS")
print("=" * 70)

round_0 = df[df['Round'] == 0]
print(f"\nTotal Round 0 rows: {len(round_0):,}")
print(f"Percentage of dataset: {len(round_0)/len(df)*100:.2f}%")

print("\n📊 Distribution by Year:")
print(round_0['Year'].value_counts().sort_index())

print("\n📊 Sample Round 0 records:")
print(round_0[['College_Code', 'College_Name', 'Branch', 'Year', 'Round', 'Cutoff_Rank']].head(10))

ROUND 0 (MOCK ALLOTMENT) ANALYSIS

Total Round 0 rows: 83,344
Percentage of dataset: 27.72%

📊 Distribution by Year:
Year
2020    10293
2021    15143
2022    16179
2023    18020
2024    23709
Name: count, dtype: int64

📊 Sample Round 0 records:
  College_Code                                       College_Name    Branch  \
0         E001  University Visveswariah College of Engineering...  CE Civil   
1         E001  University Visveswariah College of Engineering...  CE Civil   
2         E001  University Visveswariah College of Engineering...  CE Civil   
3         E001  University Visveswariah College of Engineering...  CE Civil   
4         E001  University Visveswariah College of Engineering...  CE Civil   
5         E001  University Visveswariah College of Engineering...  CE Civil   
6         E001  University Visveswariah College of Engineering...  CE Civil   
7         E001  University Visveswariah College of Engineering...  CE Civil   
8         E001  University Visveswariah Coll

## Step 2: Analyze MISSING_COLLEGE Data

Before removing MISSING_COLLEGE rows, let's investigate:
- How many MISSING_COLLEGE rows exist?
- Which college codes have MISSING_COLLEGE?
- **CRITICAL**: Do these college codes have proper names in other years/rounds?

In [3]:
# Analyze MISSING_COLLEGE data
print("=" * 70)
print("MISSING_COLLEGE ANALYSIS")
print("=" * 70)

missing_college = df[df['College_Name'] == 'MISSING_COLLEGE']
print(f"\nTotal MISSING_COLLEGE rows: {len(missing_college):,}")
print(f"Percentage of dataset: {len(missing_college)/len(df)*100:.2f}%")

print("\n📊 Distribution by Year:")
print(missing_college['Year'].value_counts().sort_index())

print("\n📊 Distribution by Round:")
print(missing_college['Round'].value_counts().sort_index())

missing_codes = sorted(missing_college['College_Code'].unique())
print(f"\n📊 Unique College Codes with MISSING_COLLEGE: {len(missing_codes)}")
print(f"Sample codes: {missing_codes[:20]}")

print("\n" + "=" * 70)
print("⚠️  CRITICAL CHECK: Do these codes have proper names elsewhere?")
print("=" * 70)

MISSING_COLLEGE ANALYSIS

Total MISSING_COLLEGE rows: 3,067
Percentage of dataset: 1.02%

📊 Distribution by Year:
Year
2020    608
2021    634
2022    678
2023    704
2024    443
Name: count, dtype: int64

📊 Distribution by Round:
Round
0    780
1    773
2    755
3    759
Name: count, dtype: int64

📊 Unique College Codes with MISSING_COLLEGE: 118
Sample codes: ['E010', 'E019', 'E020', 'E021', 'E022', 'E024', 'E025', 'E026', 'E027', 'E030', 'E035', 'E039', 'E041', 'E050', 'E051', 'E052', 'E053', 'E054', 'E059', 'E062']

⚠️  CRITICAL CHECK: Do these codes have proper names elsewhere?


In [4]:
# Check if MISSING_COLLEGE codes have proper names in other records
analysis_results = []

for code in missing_codes:
    code_df = df[df['College_Code'] == code]
    total_records = len(code_df)
    missing_count = len(code_df[code_df['College_Name'] == 'MISSING_COLLEGE'])
    proper_names = code_df[code_df['College_Name'] != 'MISSING_COLLEGE']['College_Name'].unique()
    
    analysis_results.append({
        'College_Code': code,
        'Total_Records': total_records,
        'Missing_Count': missing_count,
        'Has_Proper_Name': len(proper_names) > 0,
        'Proper_Names': list(proper_names) if len(proper_names) > 0 else []
    })

analysis_df = pd.DataFrame(analysis_results)

print(f"\n✅ Codes with proper names elsewhere: {analysis_df['Has_Proper_Name'].sum()}")
print(f"❌ Codes that are ALWAYS missing: {(~analysis_df['Has_Proper_Name']).sum()}")

print("\n📋 Sample codes with proper names found:")
has_names = analysis_df[analysis_df['Has_Proper_Name'] == True].head(10)
for _, row in has_names.iterrows():
    print(f"\n{row['College_Code']}: {row['Total_Records']} records, {row['Missing_Count']} missing")
    print(f"  ✓ Proper name(s): {row['Proper_Names'][:2]}")

print("\n📋 Codes that are ALWAYS MISSING_COLLEGE:")
always_missing = analysis_df[analysis_df['Has_Proper_Name'] == False]
print(always_missing['College_Code'].tolist())


✅ Codes with proper names elsewhere: 68
❌ Codes that are ALWAYS missing: 50

📋 Sample codes with proper names found:

E010: 112 records, 32 missing
  ✓ Proper name(s): ['Islamia Institute of Technology Bangalore']

E021: 2446 records, 6 missing
  ✓ Proper name(s): ['JSS Science and Technology University(Formerly SJCE) Mysore', 'Sri Jayachamarajendra College of Engineering(Const. of JSS Univ.) Mysore']

E022: 1277 records, 2 missing
  ✓ Proper name(s): ['The National Institute of Engineering Mysore', 'The National Institute of Engineering(SOUTH CAMPUS) Mysore']

E024: 1098 records, 1 missing
  ✓ Proper name(s): ['Malnad College of Engineering Hassan', 'Malnad College of Engineering  Hassan']

E030: 869 records, 16 missing
  ✓ Proper name(s): ['KLE Technological University(Formerly BVBCET) Hubli']

E035: 331 records, 5 missing
  ✓ Proper name(s): ['Anjuman Engineering College Bhatkala, Uttar kannada Dist']

E041: 1339 records, 1 missing
  ✓ Proper name(s): ['P D A College of Engineering

## ⚠️ CONFIRMATION REQUIRED

**Please review the analysis above before proceeding with deletion!**

The next cells will:
1. Remove all Round 0 rows
2. Remove all MISSING_COLLEGE rows

**Rows to be deleted:**
- Round 0: ~83,344 rows (~27.72%)
- MISSING_COLLEGE: ~3,067 rows (~1.02%)
- **Total deletion: ~86,411 rows (~28.74%)**

**After this deletion, you will have approximately 214,278 rows remaining.**

---

### 🔍 Key Findings to Review:
- Some MISSING_COLLEGE codes have proper names in other years/rounds
- Some codes are ALWAYS MISSING_COLLEGE (these might be invalid/discontinued colleges)

**Do you want to proceed with the deletion? If yes, run the next cells.**

## Step 3: Remove Round 0 Rows

In [5]:
# Remove Round 0 rows
print("Removing Round 0 rows...")
before_count = len(df)
df_cleaned = df[df['Round'] != 0].copy()
after_count = len(df_cleaned)
removed = before_count - after_count

print(f"✅ Removed {removed:,} Round 0 rows")
print(f"📊 Before: {before_count:,} rows")
print(f"📊 After: {after_count:,} rows")
print(f"📊 Remaining: {after_count/before_count*100:.2f}%")

Removing Round 0 rows...
✅ Removed 83,344 Round 0 rows
📊 Before: 300,689 rows
📊 After: 217,345 rows
📊 Remaining: 72.28%


## Step 4: Remove MISSING_COLLEGE Rows

In [6]:
# Remove MISSING_COLLEGE rows
print("Removing MISSING_COLLEGE rows...")
before_count = len(df_cleaned)
df_cleaned = df_cleaned[df_cleaned['College_Name'] != 'MISSING_COLLEGE'].copy()
after_count = len(df_cleaned)
removed = before_count - after_count

print(f"✅ Removed {removed:,} MISSING_COLLEGE rows")
print(f"📊 Before: {before_count:,} rows")
print(f"📊 After: {after_count:,} rows")

print("\n" + "=" * 70)
print("FINAL DATASET SUMMARY")
print("=" * 70)
print(f"Original size: {len(df):,} rows")
print(f"Final size: {len(df_cleaned):,} rows")
print(f"Total removed: {len(df) - len(df_cleaned):,} rows ({(len(df) - len(df_cleaned))/len(df)*100:.2f}%)")
print(f"Retention rate: {len(df_cleaned)/len(df)*100:.2f}%")

Removing MISSING_COLLEGE rows...
✅ Removed 2,287 MISSING_COLLEGE rows
📊 Before: 217,345 rows
📊 After: 215,058 rows

FINAL DATASET SUMMARY
Original size: 300,689 rows
Final size: 215,058 rows
Total removed: 85,631 rows (28.48%)
Retention rate: 71.52%


## Step 5: Create College Code to College Name Mapping (2024 Data)

Creating a JSON mapping file with `College_Code` → `College_Name` for 2024 year data.
This will be saved in `Version2/data/processed_data/college_mapping_2024.json`

In [7]:
# Create college mapping from 2024 data
print("Creating College Code → College Name mapping for 2024...")

# Filter 2024 data
df_2024 = df_cleaned[df_cleaned['Year'] == 2024].copy()

# Get unique college code to name mappings
# Group by College_Code and take the first non-null College_Name
college_mapping = df_2024.groupby('College_Code')['College_Name'].first().to_dict()

print(f"\n✅ Created mapping for {len(college_mapping)} unique college codes")
print(f"\nSample mappings:")
for i, (code, name) in enumerate(list(college_mapping.items())[:10]):
    print(f"  {code}: {name}")
    if i >= 9:
        break

# Save as JSON
mapping_path = os.path.join(output_dir, 'college_mapping_2024.json')
with open(mapping_path, 'w', encoding='utf-8') as f:
    json.dump(college_mapping, f, indent=2, ensure_ascii=False)

print(f"\n💾 Saved mapping to: {mapping_path}")

Creating College Code → College Name mapping for 2024...

✅ Created mapping for 248 unique college codes

Sample mappings:
  E001: University of Visvesvaraya College of Engineering Bangalore ( PUBLIC UNIV. )
  E002: S K S J T Institute of Engineering. Bangalore
  E003: B M S College of Engineering Basavanagudi,Bangalore
  E004: Dr. Ambedkar Institute Of Technology Bangalore
  E005: R. V. College of Engineering Bangalore
  E006: M S Ramaiah Institute of Technology Bangalore
  E007: Dayananda Sagar College of Engineering Bangalore
  E008: Bangalore Institute of Technology Bangalore
  E009: P E S University (Ring Road Campus) Bangalore
  E011: M V J College of Engineering Bangalore

💾 Saved mapping to: D:\Major Project\college-predictor\Version2\data\processed_data\college_mapping_2024.json


## Step 6: Verify Cleaned Data Quality

In [8]:
# Verify cleaned data
print("=" * 70)
print("DATA QUALITY VERIFICATION")
print("=" * 70)

print("\n1️⃣ Round Distribution (Round 0 should be gone):")
print(df_cleaned['Round'].value_counts().sort_index())

print("\n2️⃣ MISSING_COLLEGE Check (should be 0):")
missing_check = len(df_cleaned[df_cleaned['College_Name'] == 'MISSING_COLLEGE'])
print(f"MISSING_COLLEGE rows remaining: {missing_check}")

print("\n3️⃣ Year Distribution:")
print(df_cleaned['Year'].value_counts().sort_index())

print("\n4️⃣ Dataset Info:")
print(f"Shape: {df_cleaned.shape}")
print(f"Columns: {df_cleaned.columns.tolist()}")

print("\n5️⃣ Null Values Check:")
print(df_cleaned.isnull().sum())

print("\n6️⃣ Sample of cleaned data:")
print(df_cleaned.head(10))

DATA QUALITY VERIFICATION

1️⃣ Round Distribution (Round 0 should be gone):
Round
1    84202
2    75462
3    55394
Name: count, dtype: int64

2️⃣ MISSING_COLLEGE Check (should be 0):
MISSING_COLLEGE rows remaining: 0

3️⃣ Year Distribution:
Year
2020    26034
2021    38762
2022    43048
2023    49880
2024    57334
Name: count, dtype: int64

4️⃣ Dataset Info:
Shape: (215058, 12)
Columns: ['College_Code', 'College_Name', 'Category', 'Branch', 'Cutoff_Rank', 'Year', 'Round', 'Exam_Type', 'Rank_Scaled', 'Branch_Norm', 'College_Clean', 'College_Final']

5️⃣ Null Values Check:
College_Code       0
College_Name       0
Category           0
Branch             0
Cutoff_Rank        0
Year               0
Round              0
Exam_Type          0
Rank_Scaled        0
Branch_Norm        0
College_Clean      0
College_Final    140
dtype: int64

6️⃣ Sample of cleaned data:
      College_Code                                       College_Name  \
66457         E001  University Visveswariah College of 

## Step 7: Save Cleaned Dataset

In [9]:
# Save the cleaned dataset
output_file = os.path.join(output_dir, 'KCET_split_cleaned.csv')

print(f"Saving cleaned dataset to: {output_file}")
df_cleaned.to_csv(output_file, index=False)

print(f"\n✅ Successfully saved cleaned dataset!")
print(f"\n📊 Final Statistics:")
print(f"   - Output file: {output_file}")
print(f"   - Total rows: {len(df_cleaned):,}")
print(f"   - Total columns: {len(df_cleaned.columns)}")
print(f"   - File size: {os.path.getsize(output_file) / (1024*1024):.2f} MB")

print(f"\n📋 Files created:")
print(f"   1. {output_file}")
print(f"   2. {mapping_path}")

Saving cleaned dataset to: D:\Major Project\college-predictor\Version2\data\processed_data\KCET_split_cleaned.csv

✅ Successfully saved cleaned dataset!

📊 Final Statistics:
   - Output file: D:\Major Project\college-predictor\Version2\data\processed_data\KCET_split_cleaned.csv
   - Total rows: 215,058
   - Total columns: 12
   - File size: 48.15 MB

📋 Files created:
   1. D:\Major Project\college-predictor\Version2\data\processed_data\KCET_split_cleaned.csv
   2. D:\Major Project\college-predictor\Version2\data\processed_data\college_mapping_2024.json


## ✅ Summary

### Tasks Completed:

1. ✅ **Removed Round 0 (Mock Allotment) rows** - Approximately 83,344 rows removed
2. ✅ **Removed MISSING_COLLEGE rows** - Approximately 3,067 rows removed  
3. ✅ **Created College Mapping JSON** - `college_mapping_2024.json` with College_Code → College_Name mappings

### Output Files:

1. **Cleaned Dataset**: `Version2/data/processed_data/KCET_split_cleaned.csv`
2. **College Mapping**: `Version2/data/processed_data/college_mapping_2024.json`

### Data Reduction:
- **Original**: ~300,689 rows
- **Final**: ~214,278 rows
- **Removed**: ~86,411 rows (28.74%)
- **Retention**: 71.26%

---

**Note**: The rank normalization (Rank_Scaled) is already present in the dataset as per your context, so no changes were required for that.